### 대화 지식 그래프 메모리 ( ConversationKnowledgeGraph )

In [ ]:
%pip install langchain-community

In [2]:
import os 

from langchain_classic.memory import ConversationKGMemory


from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_teddynote import logging


load_dotenv()
logging.langsmith("test0914")


print("OpenAI 키 로드됨 : ", bool(os.getenv("OPENAI_API_KEY")))
print("LangSmith 키 로드됨 : ", bool(os.getenv("LANGSMITH_API_KEY")))
print("LangSmith 프로젝트 : ", os.getenv("LANGSMITH_PROJECT"))

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914
OpenAI 키 로드됨 :  True
LangSmith 키 로드됨 :  True
LangSmith 프로젝트 :  test0914


In [9]:
llm = ChatOpenAI(model="gpt-4o", temperature=0)

In [ ]:
%pip install networkx

In [12]:
memory = ConversationKGMemory(llm=llm, return_messages=True)

memory.save_context(
    {"input": "Say hi to Sally, she lives in Pangyo"},
    {"output":"Who is Sally?"},
)
memory.save_context(
    {"input": "Sally is a new designer at our company"},
    {"output": "Nice to meet her"},
)

In [13]:
memory.load_memory_variables({"input":"Who is Sally?"})

{'history': [SystemMessage(content='On Sally: Sally lives in Pangyo. Sally is a new designer. Sally works at our company.', additional_kwargs={}, response_metadata={})]}

- 이 메모리의 내부 프롬프트는 영어로 되어 있고, fewshot예시도 전부 영어
- 한국어 문장에서 관계를 뽑아내는 걸 gpt-4o-mini 가 자주 실패함

- ConversationKGMemory 는 실무에서 거의 안 쓰이는 실험적인 기능
- 한국어 지원도 약하고, 매번 LLM을 두 번씩 호출해서 느리고 비쌈

In [14]:
from langchain_core.prompts.prompt import PromptTemplate
from langchain_classic.chains import ConversationChain



In [15]:
template = """The following is a friendly conversation between a human and an AI. 
The AI is talkative and provides lots of specific details from its context. 
If the AI does not know the answer to a question, it truthfully says it does not know. 
The AI ONLY uses information contained in the "Relevant Information" section and does not hallucinate.

Relevant Information:

{history}

Conversation:
Human: {input}

AI:
"""

prompt = PromptTemplate(
    input_variables=["history","input"], 
    template=template
)

conversation_with_kg = ConversationChain(
    llm = llm, prompt= prompt, memory = ConversationKGMemory(llm=llm)
)

C:\Users\user\AppData\Local\Temp\ipykernel_14676\2709916987.py:21: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build a conversational agent with `langchain.agents.create_agent` and persist message history via a LangGraph checkpointer.
  conversation_with_kg = ConversationChain(


In [16]:
conversation_with_kg.predict(
    input= "My name is Teddy. Shirley is a coworker of mine, and she's a new designer at our company"
)

"Hello, Teddy! It's nice to meet you. It sounds like you and Shirley are working together in a creative environment. Being a new designer, Shirley might be bringing fresh ideas and perspectives to your team. How has it been working with her so far?"

In [17]:
conversation_with_kg.memory.load_memory_variables({"input":"who is Shirley?"})

{'history': 'On Shirley: Shirley is a new designer. Shirley works at our company.'}